# ChadGPT SFT dataset cleaning (Gemma-4 12B rebuild)

Single self-contained cleaner. Run top-to-bottom. Reads the built chat SFT jsonl and writes cleaned
train/val plus a `needs_regen` worklist and a before/after report. **No network, no teacher, no GPU.**

Fixes the failure modes measured directly in the Gemma-4 eval:

1. **Name spam** — client first name repeated / re-greeted on mid-conversation turns.
   Rule: name+greeting **only** when `Is first message: yes`.
2. **Backchanneling** — call-register filler openers ("got it", "thanks for", "sure thing", ...).
   Rule: **strip all** fillers (HARD phrases always; SOFT words only before a clause boundary,
   so "Totally fine" / "Thank you for your selection" survive).
3. **Empathy (emotion-first)** — if the client's latest message is grief/loss-dominant, the reply
   must **not** pitch mortgage. Those records are **relabeled** here with empathetic, no-sell replies.
4. **Off-topic circle-back** — small-talk / mortgage-question turns should acknowledge, then return to
   the pending question. If the label doesn't circle back, we append the pending question.
5. **Anti-overfit** — de-templating (1+2) removes the most memorizable surface, plus a duplicate cap.

Anything that can't be fixed deterministically (bot-probe/guardrail, legal/compliance, degenerate-after-
strip) is routed to `needs_regen.*.jsonl` for optional higher-quality teacher re-labeling on the box.


## 1. Config

In [ ]:

import json, re, hashlib, os
from collections import Counter

# Point these at the box layout (/root/chadgpt-new/data) or a local copy.
DATA_DIR   = os.environ.get("DATA_DIR", "data")
SPLITS     = ["train", "val"]
MIN_WORDS  = 3     # a stripped suggestion shorter than this -> route to regen
DUP_CAP    = 3     # max identical (context, reply) templates to keep (anti-overfit)

def paths(split):
    return dict(
        inp   = f"{DATA_DIR}/chat_{split}.jsonl",
        out   = f"{DATA_DIR}/chat_{split}.clean.jsonl",
        regen = f"{DATA_DIR}/needs_regen.{split}.jsonl",
        report= f"{DATA_DIR}/report.{split}.json",
    )
print("DATA_DIR =", DATA_DIR)

## 2. Deterministic clean — de-name + strip backchannel

In [ ]:

# ---- filler / backchannel openers ----
# HARD: unambiguous filler phrases -> always strip when they open a message.
HARD = [
    "got it", "gotcha", "no problem at all", "not a problem at all", "no worries at all",
    "no problem", "not a problem", "no worries", "sure thing", "thanks for confirming",
    "thank you for confirming", "makes sense", "understood", "i understand", "i hear you",
    "i feel you", "i get it", "i totally get it", "sounds good", "of course", "glad to hear",
    "good to know", "good to hear", "thanks for letting me know",
    "thanks for letting us know", "thanks for sharing",
    "thanks for that", "thanks for the info", "thanks for reaching out", "thanks for the update",
    "appreciate that", "appreciate you", "i appreciate that", "i appreciate it", "for sure",
    "haha", "lol",
]
# SOFT: filler as a bare opener but valid as adverb/adjective -> strip ONLY before a clause boundary.
SOFT = [
    "okay", "ok", "alright", "great", "awesome", "perfect", "wonderful", "fantastic", "nice",
    "sure", "absolutely", "totally", "yeah", "yep", "yup", "well", "so", "right", "cool",
    "love that", "love it",
]
_HARD_ALT = "|".join(sorted((re.escape(f) for f in HARD), key=len, reverse=True))
_SOFT_ALT = "|".join(sorted((re.escape(f) for f in SOFT), key=len, reverse=True))
# empty-acknowledgment openers: "great/nice/awesome to hear (that)." — a SHORT standalone clause.
# The {0,20} char cap keeps CONTENTFUL acks ("Awesome to hear you're buying your first place").
_ACK_ADJ  = "great|nice|awesome|good|glad|happy|wonderful|perfect|lovely|excellent|amazing"
_ACK_VERB = "hear|know|talk|see|meet|connect|chat|catch up"
# Leading char class is [\s + punctuation] so a filler left BEHIND orphaned punctuation
# (e.g. "Got it. I understand." after de-naming) still gets peeled on the next loop pass.
# The filler group is required, so bare punctuation with no filler after it is never eaten.
FILLER_OPENER_RE = re.compile(
    rf"^[\s.,;:!?—-]*(?:"
    rf"(?:{_HARD_ALT})\b[\s,]*[-—]?\s*"
    rf"|(?:{_SOFT_ALT})\b\s*[,.!?—-]+\s*"
    rf"|(?:{_ACK_ADJ})\s+to\s+(?:{_ACK_VERB})\b[^.!?]{{0,20}}[.!,]+\s*"
    rf")",
    re.I,
)

def strip_name(msg, name, is_first):
    """Remove greeting+name and vocative name uses on non-first turns."""
    if is_first or not name or name.lower() in ("n/a", ""):
        return msg
    nm = re.escape(name)
    msg = re.sub(rf"^\s*(hi|hey|hello|hiya|heya)\s+{nm}\b[\s,!.:-]*", "", msg, flags=re.I)
    msg = re.sub(rf"^\s*{nm}\s*[,!:-]+\s*", "", msg)
    msg = re.sub(rf"\s*,\s*{nm}\b(?=[\s.!?,]|$)", "", msg)
    msg = re.sub(rf"\s+{nm}\s*([.!?])\s*$", r"\1", msg)
    return msg

def strip_fillers(msg):
    prev = None
    while prev != msg:
        prev = msg
        msg = FILLER_OPENER_RE.sub("", msg)
    return msg

def tidy(msg):
    msg = re.sub(r"^[\s.,;:!?—-]+", "", msg)          # drop orphaned leading punctuation
    msg = msg.strip(" \t—-,;:")
    msg = re.sub(r"\s+([.,;:!?])", r"\1", msg)             # no space before punctuation
    msg = re.sub(r"\s{2,}", " ", msg).strip()
    if msg:
        for i, ch in enumerate(msg):
            if ch.isalpha():
                msg = msg[:i] + ch.upper() + msg[i+1:]
                break
    return msg

def clean_suggestion(msg, name, is_first):
    return tidy(strip_fillers(strip_name(msg, name, is_first)))

def word_count(s):
    return len(re.findall(r"\w+", s))

## 3. Empathy — emotion-first relabel

When the client's latest message is grief/loss-**dominant** (and not tangled with loan logistics, e.g. explaining who inherited a property), replace the 3 suggestions with empathetic, no-sell replies. A pool of 8 lines with a hash-picked distinct-3 gives 56 combinations, so we don't stamp the same template everywhere.

In [ ]:

# strong loss cue in the CLIENT'S LATEST MESSAGE only
GRIEF_RE = re.compile(
    r"\b(passed away|passed on|lost my (wife|husband|spouse|mom|mother|dad|father|son|"
    r"daughter|child|partner|brother|sister)|my (wife|husband|spouse|mom|mother|dad|father|"
    r"son|daughter|partner) (just )?(died|passed)|funeral|in hospice|terminally ill|"
    r"just (died|passed)|he died|she died|they died)\b",
    re.I,
)
# if the message ALSO carries loan/property logistics, grief may be incidental -> let teacher nuance it
LOGISTICS_RE = re.compile(
    r"\b(property|deed|survivorship|title|payment|refinance|mortgage|loan|escrow|closing|"
    r"pre-?approv|rate|down payment|inherit|estate)\b", re.I,
)
PITCH_RE = re.compile(
    r"\b(mortgage|pre-?approv|\brate\b|\bloan\b|refinanc|qualify|qualifying|property use|"
    r"down payment|application|apply|move forward|next step|primary residence|"
    r"investment property|bankruptc|foreclosur|late (mortgage )?payment)\b", re.I,
)

EMPATHY = [
    "I'm so sorry for your loss. Please take all the time you need - we can pick this up whenever you're ready.",
    "I'm really sorry to hear that. There's no rush at all; reach out whenever you feel up to it.",
    "That's heartbreaking - I'm so sorry. Take care of yourself, and I'm here whenever the time feels right.",
    "I'm so sorry you're going through this. Whenever you're ready, I'm here to help - no pressure at all.",
    "My deepest condolences. Please focus on yourself right now; we can talk again whenever suits you.",
    "I can't imagine how hard this is. Take whatever time you need - I'll be here when you're ready.",
    "I'm truly sorry. Don't worry about any of this for now; reach out when you're ready and I'll help however I can.",
    "Sending my condolences. There's absolutely no rush - I'm here whenever you'd like to continue.",
]

def pick3(pool, seed_text):
    """Deterministic distinct-3 selection keyed on the context (stable across runs)."""
    h = int(hashlib.md5(seed_text.encode("utf-8")).hexdigest(), 16)
    n = len(pool)
    idxs, i = [], h % n
    while len(idxs) < 3:
        if i % n not in idxs:
            idxs.append(i % n)
        i += 1 + (h % 3)
    return [pool[j] for j in idxs]

def empathy_reco(seed_text):
    msgs = pick3(EMPATHY, seed_text)
    conf = ["high", "medium", "low"]
    return {"recommendations": [{"suggested_message": m, "confidence": c} for m, c in zip(msgs, conf)]}

def is_pure_grief(latest):
    return bool(GRIEF_RE.search(latest)) and not LOGISTICS_RE.search(latest)

## 4. Off-topic circle-back

For small-talk / mortgage-question turns that have a pending question key, make sure at least one suggestion circles back. If a suggestion doesn't already ask something, append the pending question. Bot-probe / guardrail / legal / inappropriate turns are **not** templated here (spec-sensitive) — they go to `needs_regen`.

In [ ]:

QUESTIONS = {
    "property_use":           "will this place be your primary home, a second home, or an investment?",
    "bankruptcy_past_3yr":    "have you had any bankruptcies in the past 3 years?",
    "foreclosure_past_2yr":   "any foreclosures in the past 2 years?",
    "late_payments_past_12mo":"any late mortgage payments in the last 12 months?",
}

def situation_of(user_content):
    for line in user_content.splitlines():
        if line.startswith("SITUATION"):
            return line
    return ""

# situations where circling back is appropriate
CIRCLE_BACK_SIT = ("asked a mortgage question", "small talk", "small-talk", "off-topic", "off topic")
# situations we must NOT template (route to teacher regen instead)
ROUTE_SIT = ("inappropriate", "legal", "compliance", "complaint", "stop", "opt out", "opt-out")

# bot-probe guardrail leak: a reply that claims to be human. The eval flagged "I'm a real person"
# as a guardrail violation, so these labels teach the wrong thing -> route to teacher regen.
GUARDRAIL_RE = re.compile(
    r"\b(a real person|real human|not a bot|i'?m (a )?real|i am (a )?real|really a person|"
    r"i'?m human|i am human|yes,? (a )?real|100% real|a human being)\b", re.I,
)

def maybe_circle_back(msg, qkey):
    q = QUESTIONS.get(qkey)
    if not q or "?" in msg:
        return msg
    body = msg.rstrip(" .!")
    return f"{body}. Anyway - {q[0].lower()}{q[1:]}"

## 5. Assemble — one pass over each split

In [ ]:

def ctx_fields(user_content):
    d = {}
    for line in user_content.splitlines():
        if ":" in line and not line.startswith("---") and not line.startswith("SITUATION"):
            k, _, v = line.partition(":")
            d[k.strip()] = v.strip()
    return d

def process(split):
    p = paths(split)
    stats = Counter()
    kept, regen, seen = [], [], Counter()

    for line in open(p["inp"]):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        stats["total"] += 1
        msgs = rec["messages"]
        user = next(m["content"] for m in msgs if m["role"] == "user")
        asst = next(m["content"] for m in msgs if m["role"] == "assistant")
        d = ctx_fields(user)
        sit = situation_of(user)
        is_first = d.get("Is first message", "no").strip().lower() in ("yes", "true")
        name = d.get("Client name", "")
        qkey = d.get("Next question key", "").strip()
        latest = d.get("Client's latest message", "")

        try:
            obj = json.loads(asst)
            suggs = obj["recommendations"]
            assert isinstance(suggs, list) and len(suggs) == 3
        except Exception:
            stats["unparsable_dropped"] += 1
            continue

        # ---- 3. empathy: pure-grief latest message -> relabel emotion-first ----
        if is_pure_grief(latest):
            stats["grief_relabeled"] += 1
            obj = empathy_reco(user)
        else:
            # ---- grief tangled with logistics + a pitch -> teacher nuance ----
            if GRIEF_RE.search(latest) and any(PITCH_RE.search(s.get("suggested_message","")) for s in suggs):
                stats["grief_mixed_to_regen"] += 1
                regen.append({**rec, "_regen_reason": "grief_mixed_logistics"})
                continue

            # ---- spec-sensitive situations -> teacher ----
            if any(k in sit.lower() for k in ROUTE_SIT):
                stats["spec_sensitive_to_regen"] += 1
                regen.append({**rec, "_regen_reason": "spec_sensitive"})
                continue

            # ---- bot-probe guardrail leak (claims to be human) -> teacher ----
            if any(GUARDRAIL_RE.search(s.get("suggested_message", "")) for s in suggs):
                stats["guardrail_leak_to_regen"] += 1
                regen.append({**rec, "_regen_reason": "guardrail_humanity"})
                continue

            # ---- 2. de-name + strip backchannel ----
            new = []
            degenerate = False
            for s in suggs:
                c = clean_suggestion(s.get("suggested_message", ""), name, is_first)
                if word_count(c) < MIN_WORDS:
                    degenerate = True
                    break
                new.append({**s, "suggested_message": c})
            if degenerate:
                stats["degenerate_to_regen"] += 1
                regen.append({**rec, "_regen_reason": "degenerate_after_strip"})
                continue

            # ---- 4. off-topic circle-back ----
            if any(k in sit.lower() for k in CIRCLE_BACK_SIT) and qkey and \
               not any("?" in s["suggested_message"] for s in new):
                new[0]["suggested_message"] = maybe_circle_back(new[0]["suggested_message"], qkey)
                stats["circle_back_added"] += 1

            if len({s["suggested_message"] for s in new}) < 3:
                stats["collapsed_to_regen"] += 1
                regen.append({**rec, "_regen_reason": "collapsed_duplicates"})
                continue

            obj["recommendations"] = new

        new_asst = json.dumps(obj, ensure_ascii=False)
        if new_asst != asst:
            stats["records_modified"] += 1
        rec = {**rec, "messages": [
            m if m["role"] != "assistant" else {**m, "content": new_asst} for m in msgs
        ]}

        # ---- anti-overfit: cap identical (context, reply) templates ----
        key = (user, new_asst)
        seen[key] += 1
        if seen[key] > DUP_CAP:
            stats["dup_capped"] += 1
            continue
        kept.append(rec)

    with open(p["out"], "w") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(p["regen"], "w") as f:
        for r in regen:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    report = dict(stats); report["kept"] = len(kept); report["routed_to_regen"] = len(regen)
    with open(p["report"], "w") as f:
        json.dump(report, f, indent=2)
    return report

for split in SPLITS:
    print(f"=== {split} ===")
    print(json.dumps(process(split), indent=2))

## 6. Verify — before/after metrics

Sanity-check the cleaning actually moved the numbers and left no artifacts.

In [ ]:

greet  = re.compile(r"^\s*(hi|hey|hello|hiya|heya)\b", re.I)
filler = re.compile(r"^\s*(got it|gotcha|thanks for|no problem|sure thing|i understand|understood|"
                    r"great|absolutely|no worries|yeah|awesome|totally|sounds good|nice|of course|"
                    r"perfect|makes sense|good to)\b", re.I)

def audit(path):
    recs = [json.loads(l) for l in open(path)]
    tot = nf = fill = sug = name_nf = greet_nf = artifacts = 0
    for r in recs:
        u = next(m["content"] for m in r["messages"] if m["role"] == "user")
        a = next(m["content"] for m in r["messages"] if m["role"] == "assistant")
        d = ctx_fields(u)
        try: ss = [x["suggested_message"] for x in json.loads(a)["recommendations"]]
        except: continue
        tot += 1
        isf = d.get("Is first message", "no").lower() in ("yes", "true")
        nm = d.get("Client name", "")
        for s in ss:
            sug += 1
            if filler.match(s): fill += 1
            if re.match(r"^[\s.,;:!?]", s) or "  " in s: artifacts += 1
        if not isf:
            nf += 1
            if nm and nm.lower() not in ("n/a", "") and any(nm.lower() in s.lower() for s in ss): name_nf += 1
            if any(greet.match(s) for s in ss): greet_nf += 1
    return tot, fill, sug, name_nf, greet_nf, nf, artifacts

for split in SPLITS:
    p = paths(split)
    for label, path in [("BEFORE", p["inp"]), ("AFTER ", p["out"])]:
        tot, fill, sug, name_nf, greet_nf, nf, art = audit(path)
        print(f"[{split}] {label} recs={tot:5}  filler={fill/max(sug,1):.1%}  "
              f"name-on-nonfirst={name_nf/max(nf,1):.1%}  greet-on-nonfirst={greet_nf/max(nf,1):.1%}  "
              f"artifacts={art}")
    print()

## Notes

- **What's fully fixed here (deterministic, no teacher):** name spam, backchannel, empathy relabel for
  pure-grief turns, off-topic circle-back append.
- **What still benefits from teacher regen** (written to `needs_regen.*.jsonl`): grief tangled with loan
  logistics, spec-sensitive turns (bot-probe/guardrail, legal/compliance, stop/opt-out), and any record
  that degenerated after stripping. Re-label those on the box with the emotion-first + guardrail prompt
  rules, then concatenate onto `chat_*.clean.jsonl`.
- **Overfitting:** de-templating (name/backchannel) removes the most memorizable surface; `DUP_CAP` bounds
  identical templates; the empathy pool yields 56 distinct triples. On the training side for Gemma-4 12B:
  LoRA r16/alpha32, dropout ~0.05-0.1, keep eval-every-100 + early-stop patience 3, cap 2-3 epochs and
  take the best checkpoint - 12B overfits ~12k rows faster than 26B did.
- Point the training notebook's data paths at `chat_train.clean.jsonl` / `chat_val.clean.jsonl`.
